# 02 — Pre-Churn Analysis and Feature Store

This notebook analyzes tenants that churned before the telemetry cutoff to understand what changes before churn and how long before contract renewal tenants typically churn.  
Those insights are then used to build a health score and action flags for Customer Success.  

The final output of this notebook is the central analytical table used for further modeling and dashboarding. It is built at the tenant grain for the 2024-06-30 snapshot and uses only information available on or before that date.


In [7]:

from pathlib import Path
import sys
import pandas as pd
import numpy as np

TELEMETRY_END_DATE = pd.Timestamp("2024-06-29")
SNAPSHOT_DATE = pd.Timestamp("2024-06-30")

PROJECT_ROOT = Path.cwd().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT / "src"))

import syskit_utils as su

DB_PATH = "saas_dataset.sqlite"
TABLES = su.load_tables(DB_PATH)


In [8]:
def get_observable_churners(
    subscriptions: pd.DataFrame,
    telemetry_end_date: pd.Timestamp = TELEMETRY_END_DATE,
) -> pd.DataFrame:
    churners = subscriptions.loc[
        (subscriptions["churned"] == 1) & (subscriptions["churn_date"] < telemetry_end_date + pd.Timedelta(days=1))
    ].copy()
    churners["days_before_renewal_churn"] = (
        churners["next_renewal_date"] - churners["churn_date"]
    ).dt.days
    return churners


def prechurn_event_drops(
    events: pd.DataFrame, churners: pd.DataFrame
) -> pd.DataFrame:
    tmp = events.merge(churners[["tenant_id", "churn_date"]], on="tenant_id", how="inner")
    tmp["days_to_churn"] = (tmp["churn_date"] - tmp["event_date"]).dt.days
    tmp = tmp.loc[(tmp["days_to_churn"] >= 0) & (tmp["days_to_churn"] <= 90)].copy()
    tmp["window"] = np.where(tmp["days_to_churn"] <= 45, "0_45", "46_90")
    agg = (
        tmp.groupby(["tenant_id", "event_name", "window"])["event_count"]
        .sum()
        .unstack(fill_value=0)
        .reset_index()
    )
    agg["avg_daily_0_45"] = agg.get("0_45", 0) / 30
    agg["avg_daily_46_90"] = agg.get("46_90", 0) / 30
    agg["abs_change"] = agg["avg_daily_0_45"] - agg["avg_daily_46_90"]
    agg["pct_change"] = np.where(
        agg["avg_daily_46_90"] > 0,
        agg["avg_daily_0_45"] / agg["avg_daily_46_90"] - 1,
        np.nan,
    )
    summary = (
        agg.groupby("event_name")[["avg_daily_0_45", "avg_daily_46_90", "abs_change", "pct_change"]]
        .mean()
        .sort_values("pct_change")
        .reset_index()
    )
    return summary

In [9]:
subscriptions_clean = pd.read_csv(PROJECT_ROOT / "data" / "subscriptions_clean.csv", parse_dates=[
    "contract_start_date", "renewal_date", "churn_date", "next_renewal_date"
])

churners = get_observable_churners(subscriptions_clean)
event_drops = prechurn_event_drops(TABLES["events"], churners)

print(f"Observable churners before telemetry end: {len(churners)}")
display(churners["days_before_renewal_churn"].describe(percentiles=[0.25, 0.5, 0.75]).to_frame("days_before_renewal_churn"))
display(event_drops)


Observable churners before telemetry end: 50


,days_before_renewal_churn
count,50.000000
mean,46.200000
std,30.065914
min,2.000000
25%,16.750000
50%,45.500000
75%,75.750000
max,87.000000


window,event_name,avg_daily_0_45,avg_daily_46_90,abs_change,pct_change
0,sensitivity_label_applied,0.189744,1.256410,-1.066667,-0.870481
1,license_recommendation_applied,0.197297,1.336937,-1.139640,-0.818447
2,pp_sync_completed,0.185000,1.190000,-1.005000,-0.766147
3,risky_workspace_resolved,0.185366,1.051220,-0.865854,-0.751352
4,policy_created,0.181579,1.230702,-1.049123,-0.735548
5,report_generated,0.681197,1.186325,-0.505128,-0.325142
6,policy_updated,0.640171,1.152991,-0.512821,-0.322793
7,risky_workspace_detected,0.610476,1.163810,-0.553333,-0.308982
8,license_optimization_viewed,0.769298,1.214035,-0.444737,-0.248681
9,pp_sync_started,0.712281,1.303509,-0.591228,-0.207938


## Key takeaways

1. Churn tends to occur **before** contract expiration, rather than exactly at the renewal date.
2. Usage declines across all events before churn, but some events drop by more than 70% (`sensitivity_label_applied`, `license_recommendation_applied`, `pp_sync_completed`, `risky_workspace_resolved`, `policy_created`), while other events decline by less than 33%.  
   In downstream analysis, these high-signal events are treated as `VALUE_EVENTS` and given greater weight for building health score.


In [10]:
VALUE_EVENTS = [
    "sensitivity_label_applied",
    "license_recommendation_applied",
    "pp_sync_completed",
    "risky_workspace_resolved",
    "policy_created",
]

OTHER_EVENTS = [
    "report_generated",
    "policy_updated",
    "risky_workspace_detected",
    "license_optimization_viewed",
    "pp_sync_started",
    "sensitivity_label_recommended",
]

CRM_ACTIVITY_TYPES = [
    "email_sent",
    "call_completed",
    "meeting_held",
    "support_ticket",
    "qbr_completed",
]

CRM_OUTCOMES = ["positive", "neutral", "negative", "no_response"]

def build_feature_store(
    tables: dict,
    subscriptions = pd.DataFrame,
    snapshot_date: pd.Timestamp = SNAPSHOT_DATE,
    value_events: list = VALUE_EVENTS,
    other_events: list = OTHER_EVENTS
) -> pd.DataFrame:
    tenants = tables["tenants"].copy()
    users = tables["users"].copy()
    events = tables["events"].copy()
    crm_companies = tables["crm_companies"].copy()
    crm_activities = tables["crm_activities"].copy()

    base = (
        tenants.merge(subscriptions, on="tenant_id", suffixes=("", "_sub"))
        .merge(crm_companies, on="tenant_id", suffixes=("", "_crm"))
        .copy()
    )
    base["snapshot_active"] = (
        (base["churned"] == 0) | (base["churn_date"] >= snapshot_date)
    ).astype(int)
    base["churn_90d"] = (
        (base["churned"] == 1)
        & (base["churn_date"] > snapshot_date)
        & (base["churn_date"] <= snapshot_date + pd.Timedelta(days=90))
    ).astype(int)
    base["days_to_renewal"] = (base["next_renewal_date"] - snapshot_date).dt.days
    base["tenure_days"] = (snapshot_date - base["contract_start_date"]).dt.days
    base["days_to_renewal_<180"] = base["days_to_renewal"] < 180

    uc = users.copy()
    user_agg = (
        uc.groupby("tenant_id")
        .agg(
            users_total=("user_id", "nunique"),
            admins=("role", lambda s: int((s == "admin").sum())),
            members=("role", lambda s: int((s == "member").sum())),
            read_only=("role", lambda s: int((s == "read-only").sum()))
        )
        .reset_index()
    )
    feature_store = base.merge(user_agg, on="tenant_id", how="left")

    ev = events.loc[events["event_date"] <= snapshot_date].copy()
    event_names = sorted(ev["event_name"].unique().tolist())

    for window in [30, 60, 90]:
        start = snapshot_date - pd.Timedelta(days=window)
        ev_w = ev.loc[(ev["event_date"] > start) & (ev["event_date"] <= snapshot_date)].copy()

        agg = (
            ev_w.groupby("tenant_id")
            .agg(
                **{
                    f"events_{window}d": ("event_count", "sum"),
                    f"active_event_days_{window}d": ("event_date", "nunique"),
                    f"active_users_evt_{window}d": ("user_id", "nunique"),
                    f"event_types_used_{window}d": ("event_name", "nunique"),
                }
            )
            .reset_index()
        )
        feature_store = feature_store.merge(agg, on="tenant_id", how="left")
        feature_store[[f"events_{window}d", f"active_event_days_{window}d", f"active_users_evt_{window}d", f"event_types_used_{window}d"]] = feature_store[[f"events_{window}d", f"active_event_days_{window}d", f"active_users_evt_{window}d", f"event_types_used_{window}d"]].fillna(0)

        piv = pd.pivot_table(
            ev_w,
            index="tenant_id",
            columns="event_name",
            values="event_count",
            aggfunc="sum",
            fill_value=0,
        )
        piv.columns = [f"evt_{c}_{window}d" for c in piv.columns]
        feature_store = feature_store.merge(piv.reset_index(), on="tenant_id", how="left")

    cur45 = ev.loc[
        (ev["event_date"] > snapshot_date - pd.Timedelta(days=45))
        & (ev["event_date"] <= snapshot_date)
    ].copy()
    prev45 = ev.loc[
        (ev["event_date"] > snapshot_date - pd.Timedelta(days=90))
        & (ev["event_date"] <= snapshot_date - pd.Timedelta(days=45))
    ].copy()

    cur_agg = (
        cur45.groupby("tenant_id")
        .agg(cur45_events=("event_count", "sum"), cur45_active_users=("user_id", "nunique"))
        .reset_index()
    )
    prev_agg = (
        prev45.groupby("tenant_id")
        .agg(prev45_events=("event_count", "sum"), prev45_active_users=("user_id", "nunique"))
        .reset_index()
    )
    feature_store = feature_store.merge(cur_agg, on="tenant_id", how="left").merge(prev_agg, on="tenant_id", how="left")
    feature_store["trend45_total_events"] = (
        feature_store["cur45_events"].fillna(0) - feature_store["prev45_events"].fillna(0)
    ) / (feature_store["prev45_events"].fillna(0) + 1)
    feature_store["trend45_active_users"] = (
        feature_store["cur45_active_users"].fillna(0) - feature_store["prev45_active_users"].fillna(0)
    ) / (feature_store["prev45_active_users"].fillna(0) + 1)


    cur45_value = ev.loc[
        (ev["event_date"] > snapshot_date - pd.Timedelta(days=45))
        & (ev["event_date"] <= snapshot_date)
        & (ev["event_name"].isin(value_events))
    ].copy()
    prev45_value = ev.loc[
        (ev["event_date"] > snapshot_date - pd.Timedelta(days=90))
        & (ev["event_date"] <= snapshot_date - pd.Timedelta(days=45))
        & (ev["event_name"].isin(value_events))
    ].copy()

    cur_agg_value = (
        cur45_value.groupby("tenant_id")
        .agg(cur45_events_value=("event_count", "sum"), cur45_active_users_value=("user_id", "nunique"))
        .reset_index()
    )
    prev_agg_value = (
        prev45_value.groupby("tenant_id")
        .agg(prev45_events_value=("event_count", "sum"), prev45_active_users_value=("user_id", "nunique"))
        .reset_index()
    )
    feature_store = feature_store.merge(cur_agg_value, on="tenant_id", how="left").merge(prev_agg_value, on="tenant_id", how="left")
    feature_store["trend45_value_events"] = (
        feature_store["cur45_events_value"].fillna(0) - feature_store["prev45_events_value"].fillna(0)
    ) / (feature_store["prev45_events_value"].fillna(0) + 1)
    feature_store["trend45_active_users_value"] = (
        feature_store["cur45_active_users_value"].fillna(0) - feature_store["prev45_active_users_value"].fillna(0)
    ) / (feature_store["prev45_active_users_value"].fillna(0) + 1)


    for event_name in event_names:
        cur_s = (
            cur45.loc[cur45["event_name"] == event_name]
            .groupby("tenant_id")["event_count"]
            .sum()
            .rename(f"cur45_{event_name}")
        )
        prev_s = (
            prev45.loc[prev45["event_name"] == event_name]
            .groupby("tenant_id")["event_count"]
            .sum()
            .rename(f"prev45_{event_name}")
        )
        feature_store = feature_store.merge(cur_s.reset_index(), on="tenant_id", how="left")
        feature_store = feature_store.merge(prev_s.reset_index(), on="tenant_id", how="left")
        feature_store[f"trend45_{event_name}"] = (
            feature_store[f"cur45_{event_name}"].fillna(0)
            - feature_store[f"prev45_{event_name}"].fillna(0)
        ) / (feature_store[f"prev45_{event_name}"].fillna(0) + 1)

    for window in [30, 60, 90]:
        feature_store[f"value_events_{window}d"] = feature_store[
            [f"evt_{e}_{window}d" for e in value_events if f"evt_{e}_{window}d" in feature_store.columns]
        ].sum(axis=1)
        feature_store[f"other_events_{window}d"] = feature_store[
            [f"evt_{e}_{window}d" for e in other_events if f"evt_{e}_{window}d" in feature_store.columns]
        ].sum(axis=1)
        feature_store[f"value_share_{window}d"] = feature_store[f"value_events_{window}d"] / (
            feature_store[f"events_{window}d"].fillna(0) + 1
        )


    ca = crm_activities.loc[crm_activities["activity_date"] <= snapshot_date].copy()
    for window in [30, 60, 90, 180]:
        start = snapshot_date - pd.Timedelta(days=window)
        ca_w = ca.loc[(ca["activity_date"] > start) & (ca["activity_date"] <= snapshot_date)].copy()

        agg = (
            ca_w.groupby("tenant_id")
            .agg(
                **{
                    f"crm_touches_{window}d": ("activity_id", "count"),
                    f"crm_touch_days_{window}d": ("activity_date", "nunique"),
                    f"crm_activity_types_{window}d": ("activity_type", "nunique"),
                    f"avg_days_to_renewal_touch_{window}d": ("days_to_renewal", "mean"),
                }
            )
            .reset_index()
        )
        feature_store = feature_store.merge(agg, on="tenant_id", how="left")

        for col in CRM_OUTCOMES:
            tmp = (
                ca_w.assign(is_col=(ca_w["outcome"] == col).astype(int))
                .groupby("tenant_id")["is_col"]
                .sum()
                .rename(f"{col}_touches_{window}d")
                .reset_index()
            )
            feature_store = feature_store.merge(tmp, on="tenant_id", how="left")

        for col in CRM_ACTIVITY_TYPES:
            tmp = (
                ca_w.assign(is_col=(ca_w["activity_type"] == col).astype(int))
                .groupby("tenant_id")["is_col"]
                .sum()
                .rename(f"{col}_{window}d")
                .reset_index()
            )
            feature_store = feature_store.merge(tmp, on="tenant_id", how="left")

    last_touch = ca.groupby("tenant_id")["activity_date"].max().rename("last_crm_touch_date").reset_index()
    feature_store = feature_store.merge(last_touch, on="tenant_id", how="left")
    feature_store["days_since_last_crm_touch"] = (
        snapshot_date - feature_store["last_crm_touch_date"]
    ).dt.days

    for window in [30, 60, 90]:
        denom = feature_store[f"crm_touches_{window}d"].replace(0, np.nan)
        for outcome in CRM_OUTCOMES:
            feature_store[f"{outcome}_touch_rate_{window}d"] = (
                feature_store[f"{outcome}_touches_{window}d"].fillna(0) / denom
            ).fillna(0)

    # additional ratios for modeling and dashboarding
    for window in [30, 60, 90]:
        feature_store[f"active_user_rate_{window}d"] = (
            feature_store[f"active_users_evt_{window}d"].fillna(0)
            / (feature_store["users_total"].fillna(0) + 1)
        )
    feature_store["admin_share"] = feature_store["admins"].fillna(0) / (feature_store["users_total"].fillna(0) + 1)
    feature_store["expansion_signal"] = (
        + 0.40 * feature_store["active_user_rate_30d"].fillna(0)
        + 0.30 * feature_store["value_share_30d"].fillna(0)
        + 0.30 * (feature_store["users_total"].fillna(0).rank(pct=True))
    )
    return feature_store





### Health Score and Action Flags Logic

The `add_health_score` function converts engagement and relationship signals into a single tenant health measure and operational flags.

- `health_score_0_100` is a weighted percentile score. It emphasizes recent value-event intensity (`trend45_value_events`), then usage breadth (`trend45_active_users`, `event_types_used_30d`), with a smaller contribution from CRM interaction quality.
- `health_segment` bins the score into interpretable categories (`critical`, `watchlist`, `stable`, `strong`) for easier prioritization.
- `at_risk_flag` highlights tenants with weak health and near-term renewal pressure, so CS can prioritize retention interventions.
- `quiet_account_flag` highlights tenants with low recent activity and low value-event momentum that have not been in recent contact with CS.
- `expansion_candidate_flag` highlights tenants with strong health and growth-like behavior that are currently in lower plan tiers, indicating upsell potential.



In [11]:
def add_health_score(model_df: pd.DataFrame) -> pd.DataFrame:
    df = model_df.copy()

    def pct_rank(series: pd.Series, ascending: bool = True) -> pd.Series:
        return series.rank(pct=True, ascending=ascending)

    value_index = (pct_rank(df["trend45_value_events"]))
    breadth_index = (
        0.55 * pct_rank(df["trend45_active_users"])
        + 0.45 * pct_rank(df["event_types_used_30d"])
    )
    relationship_index = (
        0.30 * pct_rank(df["positive_touch_rate_90d"])
        + 0.30 * (1 - pct_rank(df["negative_touch_rate_90d"]))
        + 0.20 * (1 - pct_rank(df["no_response_touch_rate_90d"]))
    )

    df["health_score_0_100"] = (
        100
        * (
            0.6 * value_index
            + 0.35 * breadth_index
            + 0.05 * relationship_index
        )
    ).round(1)

    df["health_segment"] = pd.cut(
        df["health_score_0_100"],
        bins=[-np.inf, 25, 50, 75, np.inf],
        labels=["critical", "watchlist", "stable", "strong"],
    )
    df["renewal_bucket"] = pd.cut(
        df["days_to_renewal"],
        bins=[-np.inf, 30, 90, 180, np.inf],
        labels=["0-30d", "31-90d", "91-180d", "180d+"],
    )
    df["at_risk_flag"] = (
        (df["health_segment"].isin(["critical", "watchlist"]))
        & (df["days_to_renewal"] <= 90)
    ).astype(int)
    df["quiet_account_flag"] = (
        (df["active_users_evt_30d"].fillna(0) <= 1)
        & (df["crm_touches_30d"].fillna(0) == 0)
        & (df["trend45_total_events"].fillna(0) < 0)
    ).astype(int)
    df["expansion_candidate_flag"] = (
        (df["expansion_signal"] >= 0.75)
        & (df["health_score_0_100"] >= 50)
        & (df["plan"] != "enterprise")
    ).astype(int)
    return df



feature_store = build_feature_store(TABLES, subscriptions_clean)
feature_store = add_health_score(feature_store)
feature_store.to_csv(PROJECT_ROOT / "results" / "tenant_feature_store_snapshot_2024_06_30.csv", index=False)
event_drops.to_csv(PROJECT_ROOT / "results" / "prechurn_event_drops.csv", index=False)

feature_store.shape

(500, 208)

In [12]:
health_segment_summary = feature_store["health_segment"].value_counts().rename_axis("health_segment").to_frame("tenants")
display(health_segment_summary)

,tenants
health_segment,
watchlist,210
stable,190
critical,54
strong,46
